# Generation for baselines B0 and B1 (Google Colab, T4 GPU)

Runs **Qwen2.5-Coder-7B-Instruct** (4-bit) with greedy decoding on the prompts written by
`scripts/make_prompts.py` and saves one answer per problem, together with the log-probability
of every generated token (needed later for the uncertainty estimates of checkpoint 2).

1. `Runtime > Change runtime type > T4 GPU`
2. Run all cells; when asked, upload `data/prompts_b0.jsonl` and `data/prompts_b1.jsonl`
3. Copy the downloaded `completions_b0.jsonl` / `completions_b1.jsonl` to `results/` in the repository

The loop can be interrupted and re-run: problems already answered are skipped.

In [ ]:
!pip install -q -U "transformers>=4.44" accelerate bitsandbytes

In [ ]:
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
RUNS = ["b0", "b1"]        # one prompts_<run>.jsonl each
MAX_NEW_TOKENS = 512
USE_DRIVE = True           # keep prompts and answers in Google Drive, so a disconnect loses nothing

In [ ]:
import json, os

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/vakg"
else:
    WORK = "/content/vakg"
os.makedirs(WORK, exist_ok=True)

missing = [f"prompts_{r}.jsonl" for r in RUNS if not os.path.exists(f"{WORK}/prompts_{r}.jsonl")]
if missing:
    from google.colab import files
    print("Upload:", ", ".join(missing))
    for name, data in files.upload().items():
        with open(f"{WORK}/{name}", "wb") as f:
            f.write(data)
for r in RUNS:
    print(r, sum(1 for _ in open(f"{WORK}/prompts_{r}.jsonl")), "prompts")

In [ ]:
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
tok = AutoTokenizer.from_pretrained(MODEL)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=quant, device_map="auto")
model.eval()

eos = model.generation_config.eos_token_id
EOS_IDS = set(eos if isinstance(eos, list) else [eos])
print(torch.cuda.get_device_name(0), "| transformers", transformers.__version__,
      f"| {torch.cuda.memory_allocated() / 2**30:.1f} GB used")

In [ ]:
@torch.inference_mode()
def generate(messages):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    n_prompt = inputs.input_ids.shape[1]
    out = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS,
        # greedy decoding: always the most probable token, so the run is deterministic.
        # The model's default config samples and penalises repetition; both are switched off,
        # which also keeps the scores below equal to the model's own probabilities.
        do_sample=False, temperature=None, top_p=None, top_k=None, repetition_penalty=1.0,
        output_scores=True, return_dict_in_generate=True, pad_token_id=tok.eos_token_id)
    new = out.sequences[0, n_prompt:]
    # log P(token | prompt, previous tokens) of each generated token
    logprobs = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)[0]
    return {"answer": tok.decode(new, skip_special_tokens=True),
            "tokens": [tok.decode([t]) for t in new.tolist()],
            "logprobs": [round(x, 4) for x in logprobs.tolist()],
            "n_prompt_tokens": n_prompt,
            "finished": new[-1].item() in EOS_IDS}   # False: cut off at MAX_NEW_TOKENS

demo = generate([{"role": "user", "content": "Write a Python function that adds two numbers."}])
print(demo["answer"])
print("tokens:", len(demo["tokens"]), "| mean log-prob:", sum(demo["logprobs"]) / len(demo["logprobs"]))

In [ ]:
from tqdm.auto import tqdm

for run in RUNS:
    prompts = [json.loads(l) for l in open(f"{WORK}/prompts_{run}.jsonl")]
    out_path = f"{WORK}/completions_{run}.jsonl"
    done = {json.loads(l)["example_id"] for l in open(out_path)} if os.path.exists(out_path) else set()
    todo = [p for p in prompts if p["example_id"] not in done]
    with open(out_path, "a") as f:
        for p in tqdm(todo, desc=run):
            rec = {"example_id": p["example_id"], "run": run, "model": MODEL, **generate(p["messages"])}
            f.write(json.dumps(rec) + "\n")
            f.flush()

In [ ]:
from google.colab import files

for run in RUNS:
    recs = [json.loads(l) for l in open(f"{WORK}/completions_{run}.jsonl")]
    cut = [r["example_id"] for r in recs if not r["finished"]]
    print(f"{run}: {len(recs)} answers, {len(cut)} cut off at {MAX_NEW_TOKENS} tokens {cut}")
    files.download(f"{WORK}/completions_{run}.jsonl")
print(recs[0]["answer"])